In [1]:
# ============================================================
# CELL 1 — SETUP: LOAD DATA, DEFINE LAYER 1
# Notebook: 06_binary_personalization_validation.ipynb
#
# Validates 3 personalization layers against the deployed binary
# (Normal/Stress) model:
#   Layer 1 — personal baseline calibration
#   Layer 2 — EMA continual learning
#   Layer 3 — trending-toward-stress indicator (a separate flag,
#             never overrides the hard Normal/Stress label — see
#             backend/physiological/predictor.py StressStateMachine)
#
# wesad_features_selected.csv is already binary (Normal=0/Stress=1)
# from the source — notebook 03's create_pss_labels() maps WESAD's
# TSST condition straight to Stress, no synthetic Pre-stress class
# ever created, so there's no relabeling step needed here.
# ============================================================

import numpy as np
import pandas as pd
import os
import sys
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb

COMPONENT_PATH = r"C:\Users\weha1\OneDrive\Documents\ResearchCode\ai-neuro-monitor\physiological_signals"
sys.path.insert(0, os.path.join(COMPONENT_PATH, "src"))
from utils.config import RESULTS_PATH, MODELS_PATH

csv_path = os.path.join(RESULTS_PATH, "objective1", "wesad_features_selected.csv")
df = pd.read_csv(csv_path)
feature_cols = [c for c in df.columns if c not in ['label', 'subject_id']]

X        = df[feature_cols].values
y        = df['label'].values
subjects = df['subject_id'].values

print("=" * 65)
print("OBJECTIVE 2 (BINARY) — SETUP")
print("=" * 65)
print(f"Windows: {len(y)}  (Normal={int((y==0).sum())}, Stress={int((y==1).sum())})")
print(f"Subjects: {sorted(np.unique(subjects).tolist())}")

# ── Layer 1: personal baseline calibration (unchanged from nb 06) ──
HRV_FEATURES = ['AVNN', 'MeanHR', 'SDHR', 'pNN20', 'SD1']
EDA_FEATURES = [
    'SCL_mean', 'SCL_slope', 'SCL_std',
    'EDA_mean', 'EDA_std', 'EDA_range', 'EDA_slope',
    'SCR_count', 'SCR_amp_mean', 'SCR_width_mean',
    'EDA_q10', 'EDA_q25', 'EDA_q75', 'EDA_q90',
    'deriv1_mean', 'deriv1_std', 'deriv2_mean',
    'wavelet_approx_mean', 'wavelet_approx_energy',
    'wavelet_d1_energy', 'wavelet_d2_energy', 'wavelet_d3_energy'
]
ACC_FEATURES = ['ACC_mean', 'ACC_std', 'ACC_range']


def compute_personal_baseline(X_person, y_person, n_baseline_windows=3):
    normal_mask = (y_person == 0)
    normal_windows = X_person[normal_mask]

    if len(normal_windows) == 0:
        baseline_mean = np.mean(X_person, axis=0)
        baseline_std = np.std(X_person, axis=0)
    else:
        baseline_windows = normal_windows[:n_baseline_windows]
        baseline_mean = np.mean(baseline_windows, axis=0)
        baseline_std = np.std(baseline_windows, axis=0)

    min_std = np.abs(baseline_mean) * 0.10
    min_std = np.where(min_std < 0.01, 0.01, min_std)
    baseline_std = np.where(baseline_std < min_std, min_std, baseline_std)
    return baseline_mean, baseline_std


def apply_personal_normalization(X_person, baseline_mean, baseline_std, feature_cols_list):
    X_norm = X_person.copy().astype(np.float64)
    for i, feat in enumerate(feature_cols_list):
        val, mean, std = X_person[:, i], baseline_mean[i], baseline_std[i]
        if feat in EDA_FEATURES:
            X_norm[:, i] = val / mean if abs(mean) > 0.001 else val - mean
        else:
            X_norm[:, i] = (val - mean) / std
    return X_norm


s2_mask = subjects == 2
X_s2, y_s2 = X[s2_mask], y[s2_mask]
bm, bs = compute_personal_baseline(X_s2, y_s2)
X_s2_norm = apply_personal_normalization(X_s2, bm, bs, feature_cols)

print()
print("LAYER 1 — NORMALIZED VALUES BY CLASS (S2):")
print(f"  {'Feature':20s} {'Normal':10s} {'Stress':10s} {'Type':8s}")
print("-" * 55)
for feat in ['AVNN', 'MeanHR', 'SCL_mean', 'SCR_count', 'SD1', 'EDA_mean', 'deriv1_std']:
    if feat not in feature_cols:
        continue
    idx = feature_cols.index(feat)
    ftype = 'EDA' if feat in EDA_FEATURES else 'HRV/ACC'
    vals = [X_s2_norm[y_s2 == lid, idx].mean() if (y_s2 == lid).sum() > 0 else float('nan')
            for lid in [0, 1]]
    print(f"  {feat:20s} {vals[0]:8.3f}   {vals[1]:8.3f}   {ftype}")

print()
print("Cell 1 complete")


Config loaded successfully
OBJECTIVE 2 (BINARY) — SETUP
Windows: 913  (Normal=587, Stress=326)
Subjects: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

LAYER 1 — NORMALIZED VALUES BY CLASS (S2):
  Feature              Normal     Stress     Type    
-------------------------------------------------------
  AVNN                    0.055     -0.547   HRV/ACC
  MeanHR                 -0.056      0.586   HRV/ACC
  SCL_mean                0.322      0.515   EDA
  SCR_count               1.094      2.959   EDA
  SD1                    -1.079     -1.933   HRV/ACC
  EDA_mean                0.322      0.515   EDA
  deriv1_std              0.000      0.000   EDA

Cell 1 complete


In [2]:
# ============================================================
# CELL 2 — LAYER 2: EMA CONTINUAL LEARNING (binary model)
# Same PersonalizedStressModel/EMA architecture as the earlier
# 3-class version of this analysis — only the base model, labels,
# and reported metrics are binary now (F1-Stress, no F1-prestress).
# ============================================================

model_path = os.path.join(MODELS_PATH, "best_model_obj1_v2.pkl")
scaler_path = os.path.join(MODELS_PATH, "scaler_obj1_v2.pkl")
import joblib
binary_model = joblib.load(model_path)
binary_scaler = joblib.load(scaler_path)


class PersonalizedStressModel:
    def __init__(self, base_model, feature_cols, alpha=0.2, n_baseline_windows=3):
        self.base_model = base_model
        self.feature_cols = feature_cols
        self.alpha = alpha
        self.n_baseline_windows = n_baseline_windows
        self.baseline_mean = None
        self.baseline_std = None
        self.session_count = 0

    def calibrate(self, X_session, y_session):
        bm, bs = compute_personal_baseline(X_session, y_session, self.n_baseline_windows)
        if self.baseline_mean is None:
            self.baseline_mean, self.baseline_std = bm, bs
        else:
            self.baseline_mean = self.alpha * bm + (1 - self.alpha) * self.baseline_mean
            self.baseline_std = self.alpha * bs + (1 - self.alpha) * self.baseline_std
        self.session_count += 1

    def predict(self, X_session, scaler):
        X_norm = apply_personal_normalization(X_session, self.baseline_mean, self.baseline_std, self.feature_cols)
        X_scaled = scaler.transform(X_norm)
        return self.base_model.predict(X_scaled)


def simulate_sessions(X_person, y_person, base_model, scaler, feature_cols, n_sessions=3, alpha=0.2):
    n_windows = len(X_person)
    session_size = n_windows // n_sessions
    personal_model = PersonalizedStressModel(base_model, feature_cols, alpha=alpha)
    session_results = []

    for session_idx in range(n_sessions):
        start = session_idx * session_size
        end = start + session_size if session_idx < n_sessions - 1 else n_windows
        X_sess, y_sess = X_person[start:end], y_person[start:end]
        if len(X_sess) == 0:
            continue

        personal_model.calibrate(X_sess, y_sess)
        y_pred = personal_model.predict(X_sess, scaler)

        session_results.append({
            'session': session_idx + 1,
            'n_windows': len(X_sess),
            'n_stress': int((y_sess == 1).sum()),
            'accuracy': float(np.mean(y_sess == y_pred)),
            'f1_macro': f1_score(y_sess, y_pred, average='macro', zero_division=0),
            'f1_stress': f1_score(y_sess, y_pred, labels=[1], average='macro', zero_division=0),
        })
    return session_results


results_s2 = simulate_sessions(X_s2, y_s2, binary_model, binary_scaler, feature_cols)

print("=" * 65)
print("LAYER 2 — EMA TEST ON S2 (binary)")
print("=" * 65)
print(f"{'Session':8s} {'Windows':8s} {'Stress':7s} {'Accuracy':9s} {'F1-macro':9s} {'F1-Stress':9s}")
print("-" * 55)
for r in results_s2:
    print(f"  {r['session']:6d}   {r['n_windows']:6d}   {r['n_stress']:5d}   "
          f"{r['accuracy']:7.3f}   {r['f1_macro']:7.3f}   {r['f1_stress']:7.3f}")

print()
print("Cell 2 complete")


LAYER 2 — EMA TEST ON S2 (binary)
Session  Windows  Stress  Accuracy  F1-macro  F1-Stress
-------------------------------------------------------
       1       19       0     0.474     0.321     0.000
       2       19       0     0.421     0.296     0.000
       3       21      21     0.952     0.488     0.976

Cell 2 complete


In [3]:
# ============================================================
# CELL 3 — LAYER 3: TRENDING-TOWARD-STRESS INDICATOR
#
# Deployed design (backend/physiological/predictor.py
# StressStateMachine._trend_votes) — NOT an override. It's a
# separate flag reported alongside the hard Normal/Stress label,
# so it can never hurt classification accuracy by definition. What
# matters is whether it fires correctly: quiet on stable/resting
# signals, firing on genuinely rising ones. Tested directly here
# rather than folded into a classification F1 score, since after
# this design change there IS no "before/after F1" — the label
# never changes.
# ============================================================

class TrendIndicator:
    """Same 4-indicator voting as the original 3-class trend detector,
    but exposed
    as a read-only flag (detect()) rather than apply_override()."""

    def __init__(self, window_size=5, threshold=3):
        self.window_size = window_size
        self.threshold = threshold
        self.history = []

    def update(self, features_dict):
        self.history.append(features_dict)
        if len(self.history) > self.window_size:
            self.history.pop(0)

    def detect(self):
        if len(self.history) < self.window_size:
            return False

        indicators_triggered = 0
        checks = [
            ('SCL_mean', 'rising'), ('AVNN', 'falling'),
            ('SCR_count', 'rising'), ('deriv1_std', 'rising'),
        ]
        for feat, direction in checks:
            if feat not in self.history[0]:
                continue
            vals = [w[feat] for w in self.history]
            diffs = [vals[i] - vals[i - 1] for i in range(1, len(vals))]
            if direction == 'rising' and sum(d > 0 for d in diffs) >= 3:
                indicators_triggered += 1
            elif direction == 'falling' and sum(d < 0 for d in diffs) >= 3:
                indicators_triggered += 1

        return indicators_triggered >= self.threshold


def run_scenario(name, feature_sequence):
    detector = TrendIndicator()
    fired_at = None
    for i, feat_dict in enumerate(feature_sequence):
        detector.update(feat_dict)
        if detector.detect():
            fired_at = i
            break
    print(f"  {name:30s} fired={'yes at window ' + str(fired_at) if fired_at is not None else 'no'}")
    return fired_at


print("=" * 65)
print("LAYER 3 — TREND INDICATOR SCENARIO TESTS")
print("=" * 65)

# Scenario A: stable resting signal — should NOT fire
stable = [{'SCL_mean': 3.0, 'AVNN': 800, 'SCR_count': 2, 'deriv1_std': 0.01} for _ in range(8)]
run_scenario("Stable resting signal", stable)

# Scenario B: clearly rising toward stress — SHOULD fire
rising = [{'SCL_mean': 3.0 + 0.3 * i, 'AVNN': 800 - 15 * i,
           'SCR_count': 2 + i, 'deriv1_std': 0.01 + 0.005 * i} for i in range(8)]
run_scenario("Rising toward stress", rising)

# Scenario C: noisy but no consistent direction — should NOT fire
rng = np.random.RandomState(0)
noisy = [{'SCL_mean': 3.0 + rng.uniform(-0.2, 0.2), 'AVNN': 800 + rng.uniform(-20, 20),
          'SCR_count': 2 + rng.uniform(-1, 1), 'deriv1_std': 0.01 + rng.uniform(-0.005, 0.005)}
         for _ in range(8)]
run_scenario("Noisy, no clear trend", noisy)

print()
print("Cell 3 complete — indicator behaves as expected: quiet on stable/noisy,")
print("fires only once a genuine multi-window rising pattern is present.")
print("This is exactly what predict_window() in predictor.py exercises")
print("live, feeding StressStateMachine.history from real ring windows.")


LAYER 3 — TREND INDICATOR SCENARIO TESTS
  Stable resting signal          fired=no
  Rising toward stress           fired=yes at window 4
  Noisy, no clear trend          fired=no

Cell 3 complete — indicator behaves as expected: quiet on stable/noisy,
fires only once a genuine multi-window rising pattern is present.
This is exactly what predict_window() in predictor.py exercises
live, feeding StressStateMachine.history from real ring windows.


In [4]:
# ============================================================
# CELL 4 — FULL PERSONALIZED LOSO (binary): static vs. personalized
#
# Two approaches now, not three — Layer 3 no longer changes the
# label, so there's nothing for a third "+ trend detector" row to
# show that isn't identical to "personalized". Same architecture
# as the original 3-class personalization analysis otherwise:
# retrain per fold on personally-
# normalized data, both train and test normalized before the model
# ever sees them.
# ============================================================

XGB_PARAMS = dict(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    use_label_encoder=False, eval_metric='logloss',
    random_state=42, n_jobs=-1, verbosity=0
)

unique_subjects = sorted(np.unique(subjects))
results = {'static': {'true': [], 'pred': []}, 'personal': {'true': [], 'pred': []}}

print("=" * 65)
print("OBJECTIVE 2 (BINARY) — PERSONALIZED LOSO")
print("=" * 65)

for sid in unique_subjects:
    print(f"  S{sid}...", end=" ", flush=True)

    sid_mask = subjects == sid
    X_test, y_test = X[sid_mask], y[sid_mask]
    train_mask = ~sid_mask
    X_train_all, y_train_all, s_train_all = X[train_mask], y[train_mask], subjects[train_mask]

    # Static (no personalization)
    scaler_s = StandardScaler()
    X_tr_s = scaler_s.fit_transform(X_train_all)
    X_te_s = scaler_s.transform(X_test)
    sw = compute_sample_weight('balanced', y_train_all)
    m_static = xgb.XGBClassifier(**XGB_PARAMS)
    m_static.fit(X_tr_s, y_train_all, sample_weight=sw)
    y_pred_static = m_static.predict(X_te_s)
    results['static']['true'].extend(y_test.tolist())
    results['static']['pred'].extend(y_pred_static.tolist())

    # Personalized — normalize every subject (train and test) by their own baseline
    bm_test, bs_test = compute_personal_baseline(X_test, y_test)
    X_test_norm = apply_personal_normalization(X_test, bm_test, bs_test, feature_cols)

    X_train_norm_list = []
    for train_sid in np.unique(s_train_all):
        mask_tr = s_train_all == train_sid
        X_tr_subj, y_tr_subj = X_train_all[mask_tr], y_train_all[mask_tr]
        bm_tr, bs_tr = compute_personal_baseline(X_tr_subj, y_tr_subj)
        X_train_norm_list.append(apply_personal_normalization(X_tr_subj, bm_tr, bs_tr, feature_cols))
    X_train_norm = np.vstack(X_train_norm_list)

    scaler_p = StandardScaler()
    X_tr_p = scaler_p.fit_transform(X_train_norm)
    X_te_p = scaler_p.transform(X_test_norm)
    sw_p = compute_sample_weight('balanced', y_train_all)
    m_pers = xgb.XGBClassifier(**XGB_PARAMS)
    m_pers.fit(X_tr_p, y_train_all, sample_weight=sw_p)
    y_pred_pers = m_pers.predict(X_te_p)
    results['personal']['true'].extend(y_test.tolist())
    results['personal']['pred'].extend(y_pred_pers.tolist())

    print("done")

print()
print("=" * 75)
print("RESULTS")
print("=" * 75)
print(f"{'Approach':25s} {'Accuracy':10s} {'F1-macro':10s} {'F1-Normal':10s} {'F1-Stress':10s}")
print("-" * 75)
for name, key in [('Static XGBoost', 'static'), ('Personalized baseline', 'personal')]:
    t = np.array(results[key]['true'])
    p = np.array(results[key]['pred'])
    acc = float(np.mean(t == p))
    f1_mac = f1_score(t, p, average='macro', zero_division=0)
    f1_nor = f1_score(t, p, labels=[0], average='macro', zero_division=0)
    f1_str = f1_score(t, p, labels=[1], average='macro', zero_division=0)
    print(f"  {name:23s} {acc:8.3f}   {f1_mac:8.3f}   {f1_nor:8.3f}   {f1_str:8.3f}")

t = np.array(results['personal']['true'])
p = np.array(results['personal']['pred'])
print()
print("PERSONALIZED — FULL REPORT:")
print(classification_report(t, p, target_names=['Normal', 'Stress'], digits=3))
cm = confusion_matrix(t, p)
print("CONFUSION MATRIX:")
print(f"                 Normal  Stress")
for i, row in enumerate(cm):
    lbl = ['Normal', 'Stress'][i]
    print(f"  Actual {lbl:7s}: {row[0]:6d}  {row[1]:6d}")

results_df = pd.DataFrame([
    {'approach': 'static', 'accuracy': float(np.mean(np.array(results['static']['true']) == np.array(results['static']['pred'])))},
    {'approach': 'personal', 'accuracy': float(np.mean(np.array(results['personal']['true']) == np.array(results['personal']['pred'])))},
])
results_path = os.path.join(RESULTS_PATH, "objective2", "obj2_binary_comparison.csv")
os.makedirs(os.path.dirname(results_path), exist_ok=True)
results_df.to_csv(results_path, index=False)
print(f"\nSaved: {results_path}")
print("Cell 4 complete")


OBJECTIVE 2 (BINARY) — PERSONALIZED LOSO
  S2... done
  S3... done
  S4... done
  S5... done
  S6... done
  S7... done
  S8... done
  S9... done
  S10... done
  S11... done
  S13... done
  S14... done
  S15... done
  S16... done
  S17... done

RESULTS
Approach                  Accuracy   F1-macro   F1-Normal  F1-Stress 
---------------------------------------------------------------------------
  Static XGBoost             0.851      0.842      0.879      0.805
  Personalized baseline      0.934      0.928      0.949      0.907

PERSONALIZED — FULL REPORT:
              precision    recall  f1-score   support

      Normal      0.944     0.954     0.949       587
      Stress      0.916     0.899     0.907       326

    accuracy                          0.934       913
   macro avg      0.930     0.926     0.928       913
weighted avg      0.934     0.934     0.934       913

CONFUSION MATRIX:
                 Normal  Stress
  Actual Normal :    560      27
  Actual Stress :     33   

In [5]:
# ============================================================
# CELL 5 — SWELL CROSS-DATASET VALIDATION (binary)
#
# Same 5 HRV features common to both datasets as the earlier
# 3-class version of this analysis (SWELL has no GSR). Condition
# mapping matches the binary philosophy baked into notebook 03's
# create_pss_labels(): SWELL's "time pressure" and "interruption"
# both merge into Stress — neither dataset's intermediate
# condition gets treated as a separate ground-truth class.
# ============================================================

SWELL_PATH = r"C:\Users\weha1\datasets\SWELL\archive\hrv dataset\data\final"

swell_train = pd.read_csv(os.path.join(SWELL_PATH, "train.csv"))
swell_test = pd.read_csv(os.path.join(SWELL_PATH, "test.csv"))
swell_df = pd.concat([swell_train, swell_test], ignore_index=True)

print("=" * 65)
print("SWELL CROSS-DATASET VALIDATION (binary)")
print("=" * 65)
print(f"SWELL loaded: {swell_df.shape}")

condition_map = {'no stress': 0, 'time pressure': 1, 'interruption': 1}
swell_df['label'] = swell_df['condition'].map(condition_map)
swell_df = swell_df.dropna(subset=['label'])
swell_df['label'] = swell_df['label'].astype(int)

print("\nLabel distribution after binary mapping:")
for lid, name in {0: 'Normal', 1: 'Stress'}.items():
    count = int((swell_df['label'] == lid).sum())
    print(f"  Class {lid} ({name:6s}): {count:7d} ({count/len(swell_df)*100:.1f}%)")

swell_feature_map = {'MEAN_RR': 'AVNN', 'HR': 'MeanHR', 'SDRR': 'SDHR', 'pNN25': 'pNN20', 'SD1': 'SD1'}
swell_renamed = swell_df.rename(columns=swell_feature_map)
common_features = ['AVNN', 'MeanHR', 'SDHR', 'pNN20', 'SD1']

X_swell = swell_renamed[common_features].values
y_swell = swell_renamed['label'].values

from sklearn.impute import SimpleImputer
if np.isnan(X_swell).sum() > 0 or np.isinf(X_swell).sum() > 0:
    X_swell = np.where(np.isinf(X_swell), np.nan, X_swell)
    X_swell = SimpleImputer(strategy='median').fit_transform(X_swell)

print(f"Clean SWELL shape: {X_swell.shape}")

# Train fresh on WESAD's binary labels, HRV-only features (fair
# comparison — SWELL has no GSR to match against)
wesad_common_idx = [feature_cols.index(f) for f in common_features if f in feature_cols]
X_wesad_hrv = df[feature_cols].values[:, wesad_common_idx]
y_wesad_bin = y

scaler_cross = StandardScaler()
X_wesad_scaled = scaler_cross.fit_transform(X_wesad_hrv)
X_swell_scaled = scaler_cross.transform(X_swell)

sw_cross = compute_sample_weight('balanced', y_wesad_bin)
m_cross = xgb.XGBClassifier(**XGB_PARAMS)
m_cross.fit(X_wesad_scaled, y_wesad_bin, sample_weight=sw_cross)
y_swell_pred = m_cross.predict(X_swell_scaled)

acc = float(np.mean(y_swell == y_swell_pred))
f1_mac = f1_score(y_swell, y_swell_pred, average='macro', zero_division=0)
f1_str = f1_score(y_swell, y_swell_pred, labels=[1], average='macro', zero_division=0)

print()
print("WESAD (binary, HRV-only) -> SWELL — cross-dataset generalization:")
print(f"  Accuracy:  {acc:.3f}")
print(f"  F1-macro:  {f1_mac:.3f}")
print(f"  F1-Stress: {f1_str:.3f}")
print()
print(classification_report(y_swell, y_swell_pred, target_names=['Normal', 'Stress'], digits=3))
print("Cell 5 complete — this is a directional sanity check (does the model")
print("trained on WESAD's ramp-rate features generalize to a different")
print("device/protocol's stress signal), not a claim SWELL validates ring")
print("accuracy directly — it never touches EDA, the ring's richest signal.")


SWELL CROSS-DATASET VALIDATION (binary)
SWELL loaded: (410322, 36)

Label distribution after binary mapping:
  Class 0 (Normal):  222240 (54.2%)
  Class 1 (Stress):  188082 (45.8%)
Clean SWELL shape: (410322, 5)

WESAD (binary, HRV-only) -> SWELL — cross-dataset generalization:
  Accuracy:  0.429
  F1-macro:  0.367
  F1-Stress: 0.169

              precision    recall  f1-score   support

      Normal      0.481     0.685     0.565    222240
      Stress      0.254     0.127     0.169    188082

    accuracy                          0.429    410322
   macro avg      0.368     0.406     0.367    410322
weighted avg      0.377     0.429     0.384    410322

Cell 5 complete — this is a directional sanity check (does the model
trained on WESAD's ramp-rate features generalize to a different
device/protocol's stress signal), not a claim SWELL validates ring
accuracy directly — it never touches EDA, the ring's richest signal.
